In [ ]:
import torch, torchvision
from torchvision import transforms
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [40]:
val_transforms = transforms.Compose([
    transforms.Resize((32, 32)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_set = ImageFolder(r"C:\github\dataset\archive\test", transform=val_transforms)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [41]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [42]:
num_classes = 2

model.conv1 = nn.Conv2d(in_channels=3, out_channels=64,kernel_size=3,stride=1,padding=1,bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.fc.in_features,num_classes)
)


In [43]:
model.load_state_dict(torch.load('model/best_model.pth', map_location=device))
model.eval()

C:\Users\nsyn1\AppData\Local\Temp\ipykernel_22700\1310587834.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model/best_model.pth', map

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [44]:
model = model.to(device)  # Make sure model is on device

all_predictions = []
all_labels = []

with torch.no_grad():
    for test_batch in test_loader:
        images, labels = test_batch
        images = images.to(device)
        
        # Get predictions
        predictions = model(images)
        predicted_classes = torch.argmax(predictions, dim=1)
        
        all_predictions.extend(predicted_classes.cpu().numpy())
        all_labels.extend(labels.numpy())

# Print results
import numpy as np
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

print(f'Test Accuracy: {(all_predictions == all_labels).sum() / len(all_labels):.4f}')

# Get classification report
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_predictions))

Test Accuracy: 0.9772
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     10000
           1       0.98      0.98      0.98     10000

    accuracy                           0.98     20000
   macro avg       0.98      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000

